In [35]:
from pathlib import Path
from pprint import pprint
from plant_pheno.inference import inat_client
from plant_pheno.data import DuckDbSQL, DuckDBAdapter

In [ ]:

obs_id = [182806784,398318023]
rate = 10

TARGET_TABLE_NAME = "raw.inat_api"
SOURCE_KEY = 'uuid'
OBSERVATIONS_FIELDS = {
    "id" : True,
    "photos" : True,
}

In [ ]:
with DuckDBAdapter("/home/etienne/projects/inat-phenology-cv/data/cv_raw.duckdb") as con:
    sql_api = DuckDbSQL(con, Path("/home/etienne/projects/inat-phenology-cv/queries/api/"))
    sql_api.execute("create_api_raw_table", table_name=TARGET_TABLE_NAME)

    df = sql_api.fetch_df(
        "check_existing.sql",
        source_key=SOURCE_KEY,
        source_table_name=TARGET_TABLE_NAME,
        target_table_name=TARGET_TABLE_NAME,
    )

    items = df[SOURCE_KEY].to_list() 

    if not items:
        raise 
    
    config = inat_client.EndpointConfig(
        "observations",
        write_empty_rows=True,
        fields = OBSERVATIONS_FIELDS,
        chunk_size= 200,
        per_page= 200
    )



    fetcher = inat_client.fetchers.RateLimiterFetcher(rate = 10, ignore_not_found= True)
    with inat_client.DuckDbWriter(con, TARGET_TABLE_NAME ) as writer:
        client = inat_client.make_client(config, fetcher, writer)
        x = await client.execute(obs_id)

    sql_stage = DuckDbSQL(con, Path("/home/etienne/projects/inat-phenology-cv/queries/stage"))
    sql_stage.execute("stage_inat_requests")


DBError: Binder Error: Table "s" does not have a column named "uuid"

Candidate bindings: : "raw_id"

LINE 3: LEFT JOIN raw.inat_api t ON s.uuid  = t.uuid
                                    ^ (script=fetch_missing_items)